# georisk_twin · Prueba piloto
## Librería y API en Python para operacionalizar la metodología de un Gemelo Digital Multimodal Urbano-Industrial

**Universidad de Guayaquil · Facultad de Ingeniería Industrial · Carrera de Sistemas de la Información**
**Autores:** Zambrano Mendoza Jeremy Dario · Chango Burgos Michael Alexander

---

### ¿Qué hace este notebook?
Construye **de punta a punta una versión piloto (tipo v0.5)** del paquete `georisk_twin` y comprueba, con evidencia, los cuatro criterios de aceptación del anteproyecto. Todo ocurre dentro de Colab: se escribe el código del paquete, se instala, se levanta la API, se prueba y se mide.

| Paso | Qué se hace | Objetivo / historia de usuario |
|---|---|---|
| 0–2 | Entorno, criterios de aceptación y tablero Kanban | DSR: identificación del problema y objetivos de solución |
| 3 | Estructura del paquete instalable (`pyproject.toml`) | OE-1 · HU-01 |
| 4 | **Contratos de datos Pydantic** (se diseñan primero: *API First*) | OE-1 · HU-02 |
| 5 | Contrato de módulos y registro (*plug-in*) | OE-1 · HU-05 |
| 6 | Servicios núcleo: indicadores, grafo y módulo analítico | OE-3 |
| 7 | **Mocks** de los subproductos aún no entregados | Fase 4 Kanban (bloqueos) |
| 8 | API con FastAPI y especificación OpenAPI | OE-2 · HU-03 |
| 9 | Pruebas unitarias y cobertura con `pytest` | OE-2 · HU-04 |
| 10 | Demo de integración horizontal (datos → indicadores → grafo → analítico) | OE-3 · HU-06 |
| 11 | Conectar un módulo nuevo con la plantilla de contribución | HU-05 |
| 12 | **Evaluación**: instalación limpia, rendimiento HTTP, reproducibilidad, documentación | OE-4 |
| 13 | CI con GitHub Actions y cierre de la iteración | Fase 6 Kanban |

### Advertencias metodológicas (leer antes de ejecutar)
1. **Los módulos reales aún no existen.** El anteproyecto prevé trabajar con *mocks* y contratos de datos mientras las otras tesis entregan. Los mocks de este notebook respetan los contratos, pero **sus datos son sintéticos**: sirven para demostrar la integración, no para sacar conclusiones sobre riesgo real.
2. **Códigos de módulo.** Se usan `URB-01` (datos), `URB-02` (grafos), `URB-03` (indicadores) y `URB-04` (IA) como en el ecosistema RISKTWIN. Los códigos `SI-01`, `SI-02` e `IND-SIS-01` no se asignan a un rol concreto porque el anteproyecto no lo especifica; agrégalos en `registry.py` cuando lo definan.
3. **Metas numéricas.** La cobertura ≥ 80 % y el tiempo de respuesta p95 ≤ 1500 ms son **metas propuestas para este piloto**; el anteproyecto no fija cifras. Ajústalas con tu tutor.
4. **Analítica.** Las etiquetas Alto/Medio/Bajo de la demo se derivan de los eventos sintéticos y de la criticidad del grafo, por lo que el modelo demuestra el flujo de integración, **no una predicción validada**.
5. Ejecuta las celdas **de arriba abajo**. Si cambias un archivo del paquete después de importarlo, usa *Entorno de ejecución → Reiniciar sesión* y vuelve a ejecutar.

## Paso 0 · Preparación del entorno
Colab ya trae `pydantic`, `numpy`, `pandas`, `networkx` y `scikit-learn`. Se instalan las herramientas de API y de pruebas.

In [ ]:
!pip install -q fastapi "uvicorn[standard]" httpx pytest pytest-cov requests

In [ ]:
import os, sys, json, time, subprocess, importlib
import pandas as pd
import pydantic, fastapi, sklearn, networkx, pytest

print("Python  :", sys.version.split()[0])
print("FastAPI :", fastapi.__version__, "| Pydantic:", pydantic.__version__)
print("sklearn :", sklearn.__version__, "| NetworkX:", networkx.__version__, "| pytest:", pytest.__version__)
assert int(pydantic.__version__.split(".")[0]) >= 2, "Se requiere Pydantic v2"

## Paso 1 · Identificación del problema y criterios de aceptación (DSR)
El problema: ocho subproductos de tesis se construyen en paralelo y **nadie es dueño de la capa de integración**. El artefacto propuesto es un paquete instalable con una API documentada.

Se declaran aquí los **cuatro criterios de aceptación** del anteproyecto y las metas numéricas del piloto. Una variable `EVIDENCIA` irá acumulando los resultados medidos en cada paso, y al final se contrasta con estos criterios.

In [ ]:
CRITERIOS_ACEPTACION = {
    "C1": "Paquete instalable (pip, en entorno limpio)",
    "C2": "API con documentación OpenAPI completa",
    "C3": "Demo reproducible",
    "C4": "Conexión mínima: datos + indicadores + módulo analítico",
}
META_COBERTURA_PCT = 80    # cobertura mínima de pruebas (propuesta del piloto)
META_P95_MS = 1500         # tiempo de respuesta p95 por endpoint (propuesta del piloto)

EVIDENCIA = {}             # se llena en los pasos siguientes

pd.Series(CRITERIOS_ACEPTACION, name="Criterio de aceptación").to_frame()

### Tablero Kanban del piloto
Se sigue la metodología Kanban del anteproyecto (Backlog → Por hacer → En progreso → En revisión → Bloqueado → Hecho). En este notebook cada tarjeta equivale a un paso:

| Tarjeta | Carril | Historia | Paso |
|---|---|---|:-:|
| Diseño de la estructura del paquete | Backend/API | HU-01 | 3 |
| Diseño de modelos Pydantic | Backend/API | HU-02 | 4 |
| Servicios núcleo + mocks (se mueve a *Bloqueado* si falta el módulo real) | Integración y calidad | — | 6–7 |
| Endpoints principales (FastAPI) y OpenAPI | Backend/API | HU-03 | 8 |
| Pruebas unitarias y cobertura | Integración y calidad | HU-04 | 9 |
| Integración datos + indicadores + analítico | Integración y calidad | HU-06 | 10 |
| Plantilla de contribución | Integración y calidad | HU-05 | 11 |
| Evaluación y validación automática (CI) | Integración y calidad | — | 12–13 |

## Paso 2 · Estructura del paquete (HU-01)
Se usa el diseño **`src/`**, el estándar actual de empaquetado: evita importar el código por accidente sin instalarlo y obliga a probar lo que realmente se distribuye.

```
georisk_twin_proj/
├── pyproject.toml
├── README.md
├── CONTRIBUTING.md
├── .github/workflows/ci.yml
├── src/georisk_twin/
│   ├── __init__.py
│   ├── schemas.py     ← contratos de datos (Pydantic)
│   ├── registry.py    ← contrato de módulos + registro
│   ├── servicios.py   ← lógica núcleo (indicadores, grafo, analítico, pipeline)
│   ├── mocks.py       ← datos y módulos simulados
│   └── api.py         ← endpoints FastAPI
└── tests/
```

In [ ]:
import shutil
if os.path.exists("georisk_twin_proj"):
    shutil.rmtree("georisk_twin_proj")      # partir siempre de cero (reproducible)
for d in ["georisk_twin_proj/src/georisk_twin", "georisk_twin_proj/tests", "georisk_twin_proj/.github/workflows"]:
    os.makedirs(d, exist_ok=True)
print("Carpetas creadas.")

In [ ]:
%%writefile georisk_twin_proj/pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "georisk-twin"
version = "0.5.0"
description = "Libreria y API para operacionalizar un gemelo digital multimodal de riesgo urbano-industrial"
readme = "README.md"
requires-python = ">=3.10"
dependencies = [
    "fastapi>=0.110",
    "pydantic>=2.5",
    "uvicorn>=0.27",
    "numpy",
    "pandas",
    "networkx>=3.0",
    "scikit-learn>=1.3",
]

[project.optional-dependencies]
dev = ["pytest>=7", "pytest-cov", "httpx", "requests"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]

In [ ]:
%%writefile georisk_twin_proj/src/georisk_twin/__init__.py
"""georisk_twin: capa comun de integracion del gemelo digital multimodal urbano-industrial."""

__version__ = "0.5.0"

In [ ]:
%%writefile georisk_twin_proj/README.md
# georisk_twin

Libreria y API en Python que operacionaliza la metodologia del gemelo digital multimodal
para riesgo urbano-industrial (FCI No 46). Provee una estructura comun donde se conectan
los subproductos de las tesis (datos, indicadores, grafos, IA, simulacion y visualizacion).

## Instalacion
    pip install .            # o: pip install -e ".[dev]" para desarrollo

## Levantar la API
    uvicorn georisk_twin.api:app --port 8000
    # Documentacion interactiva: http://localhost:8000/docs
    # Especificacion OpenAPI:   http://localhost:8000/openapi.json

## Pruebas
    pytest --cov=georisk_twin

## Endpoints
| Metodo | Ruta | Descripcion |
|---|---|---|
| GET  | /salud | Estado del servicio |
| GET  | /modulos | Modulos registrados |
| POST | /modulos/{codigo}/ejecutar | Ejecuta un modulo registrado |
| GET  | /mocks/eventos | Eventos sinteticos que respetan el contrato |
| POST | /datos/resumen | Valida y resume eventos |
| POST | /indicadores/sector | Indicadores por sector y tipo |
| POST | /grafos/metricas | Metricas de conectividad, centralidad y criticidad |
| POST | /analitico/riesgo | Clasificacion Alto/Medio/Bajo por sector |
| POST | /demo/integracion | Demo de integracion horizontal |

Para conectar un modulo nuevo ver CONTRIBUTING.md.

## Paso 3 · Contratos de datos con Pydantic (HU-02) — *API First*
Siguiendo **API First Design**, lo primero que se escribe es el **contrato**: qué forma tienen los datos que circulan entre módulos. Con esto los demás equipos pueden desarrollar contra la especificación sin esperar a que el paquete esté terminado.

Decisiones de diseño:
- `extra="forbid"`: un campo desconocido **se rechaza** en vez de ignorarse. Así un desajuste entre equipos (nombre distinto, campo inventado) falla de inmediato y con un mensaje claro, que es justo el problema que motiva el proyecto.
- Rangos validados (latitud, severidad, longitudes positivas) y **referencias cruzadas** en el grafo (un arco no puede apuntar a un nodo que no existe).
- Enumeraciones para tipo de evento y nivel de riesgo: una sola convención de nombres para todo el gemelo.

In [ ]:
%%writefile georisk_twin_proj/src/georisk_twin/schemas.py
"""Contratos de datos (Pydantic v2) compartidos por todos los modulos del gemelo digital."""
from __future__ import annotations

from datetime import datetime
from enum import Enum
from typing import Literal, Optional

from pydantic import BaseModel, ConfigDict, Field, model_validator


class ContratoBase(BaseModel):
    """Base de todos los contratos: rechaza campos no declarados."""

    model_config = ConfigDict(extra="forbid")


class NivelRiesgo(str, Enum):
    BAJO = "Bajo"
    MEDIO = "Medio"
    ALTO = "Alto"


class TipoEvento(str, Enum):
    ACCIDENTE = "accidente"
    DELITO = "delito"
    INUNDACION = "inundacion"


# ----------------------------------------------------------------- datos
class Coordenada(ContratoBase):
    lat: float = Field(..., ge=-90, le=90, description="Latitud en grados decimales (WGS84).")
    lon: float = Field(..., ge=-180, le=180, description="Longitud en grados decimales (WGS84).")


class Evento(ContratoBase):
    """Evento georreferenciado (accidente, delito o inundacion)."""

    id: str = Field(..., min_length=1, description="Identificador unico del evento.")
    fecha: datetime = Field(..., description="Fecha y hora del evento (ISO 8601).")
    tipo: TipoEvento
    ubicacion: Coordenada
    severidad: int = Field(1, ge=1, le=5, description="Severidad de 1 (leve) a 5 (grave).")
    sector: Optional[str] = Field(None, description="Sector, barrio o parroquia.")


class SolicitudEventos(ContratoBase):
    eventos: list[Evento] = Field(..., min_length=1)


class ResumenEventos(ContratoBase):
    n_eventos: int
    fecha_min: datetime
    fecha_max: datetime
    por_tipo: dict[str, int]
    n_sectores: int


# ---------------------------------------------------------------- grafos
class NodoGrafo(ContratoBase):
    id: int = Field(..., ge=0)
    ubicacion: Coordenada
    exposicion: float = Field(0.0, ge=0, description="Exposicion del nodo (>= 0, cualquier escala).")
    sector: Optional[str] = None


class ArcoGrafo(ContratoBase):
    origen: int = Field(..., ge=0)
    destino: int = Field(..., ge=0)
    longitud_m: float = Field(..., gt=0, description="Longitud del tramo en metros.")

    @model_validator(mode="after")
    def _sin_autorreferencia(self) -> "ArcoGrafo":
        if self.origen == self.destino:
            raise ValueError("Un arco no puede conectar un nodo consigo mismo.")
        return self


class GrafoUrbano(ContratoBase):
    nodos: list[NodoGrafo] = Field(..., min_length=2)
    arcos: list[ArcoGrafo] = Field(..., min_length=1)

    @model_validator(mode="after")
    def _referencias_validas(self) -> "GrafoUrbano":
        ids = [n.id for n in self.nodos]
        if len(set(ids)) != len(ids):
            raise ValueError("Hay identificadores de nodo duplicados.")
        conocidos = set(ids)
        for a in self.arcos:
            if a.origen not in conocidos or a.destino not in conocidos:
                raise ValueError(f"El arco {a.origen}-{a.destino} referencia un nodo inexistente.")
        return self


class MetricasNodo(ContratoBase):
    id: int
    sector: Optional[str] = None
    grado: int
    betweenness: float
    closeness: float
    exposicion_normalizada: float = Field(..., ge=0, le=1)
    criticidad: float = Field(..., ge=0, le=1)
    nivel: NivelRiesgo


class ResultadoGrafo(ContratoBase):
    n_nodos: int
    n_arcos: int
    conexo: bool
    n_componentes: int
    metricas: list[MetricasNodo]


# ------------------------------------------------------------ indicadores
class IndicadorSector(ContratoBase):
    sector: str
    tipo: TipoEvento
    n_eventos: int
    severidad_media: float
    puntaje: float = Field(..., ge=0, le=1, description="n_eventos x severidad media, normalizado a [0, 1].")
    nivel: NivelRiesgo


# -------------------------------------------------------------- analitico
class RegistroSector(ContratoBase):
    sector: str
    n_eventos: int = Field(..., ge=0)
    severidad_media: float = Field(..., ge=0)
    criticidad_media: float = Field(..., ge=0, le=1)
    exposicion_media: float = Field(..., ge=0)
    nivel_riesgo: Optional[NivelRiesgo] = Field(None, description="Etiqueta para entrenar.")


class SolicitudAnalitica(ContratoBase):
    registros: list[RegistroSector] = Field(..., min_length=6)


class PrediccionSector(ContratoBase):
    sector: str
    nivel: NivelRiesgo
    probabilidad: float = Field(..., ge=0, le=1)


class ResultadoAnalitico(ContratoBase):
    modelo: str
    n_registros: int
    exactitud_cv: Optional[float] = Field(None, description="Exactitud media en validacion cruzada.")
    predicciones: list[PrediccionSector]


# ----------------------------------------------------------- modulos / demo
class ModuloInfo(ContratoBase):
    codigo: str
    nombre: str
    version: str
    estado: Literal["mock", "real"]


class SolicitudDemo(ContratoBase):
    n_eventos: int = Field(300, ge=10, le=5000)
    semilla: int = 42


class RespuestaDemo(ContratoBase):
    modulos_usados: list[str]
    n_eventos: int
    n_nodos: int
    n_sectores: int
    indicadores_top: list[IndicadorSector]
    nodos_criticos_top: list[MetricasNodo]
    analitico: ResultadoAnalitico
    tiempos_ms: dict[str, float]
    nota: str

Primera comprobación del contrato: un evento válido, y errores típicos de integración (coordenada imposible, campo con otro nombre) que ahora se detectan con un mensaje claro.

In [ ]:
sys.path.insert(0, os.path.abspath("georisk_twin_proj/src"))
from pydantic import ValidationError
from georisk_twin.schemas import Evento

valido = {"id": "EV-1", "fecha": "2025-03-14T08:30:00", "tipo": "accidente",
          "ubicacion": {"lat": -2.24, "lon": -79.90}, "severidad": 3, "sector": "S01"}
print("✔ Válido:", Evento(**valido).model_dump(mode="json"))

casos_malos = {
    "latitud fuera de rango": {**valido, "ubicacion": {"lat": 95, "lon": -79.9}},
    "campo con otro nombre (otro equipo escribió 'gravedad')": {**{k: v for k, v in valido.items() if k != "severidad"}, "gravedad": 3},
    "tipo de evento no reconocido": {**valido, "tipo": "incendio"},
}
for nombre, dato in casos_malos.items():
    try:
        Evento(**dato)
    except ValidationError as e:
        err = e.errors()[0]
        print(f"✘ {nombre} → {err['loc']}: {err['msg']}")

## Paso 4 · Contrato de módulos y registro (HU-05)
Además de los datos, hay que acordar **cómo se conecta un módulo**. El contrato es mínimo a propósito, para no imponer restricciones sobre la implementación interna de cada tesis:

- Atributos: `codigo`, `nombre`, `version`, `estado` (`"mock"` o `"real"`).
- Un método `ejecutar(entrada: dict) -> dict` que recibe y devuelve estructuras serializables en JSON.

El `RegistroModulos` verifica que el módulo cumpla el contrato (*duck typing* con `Protocol`) y evita códigos duplicados. Cuando una tesis entregue su módulo real, se registra con el mismo código y `estado="real"` y reemplaza al *mock* sin tocar nada más.

In [ ]:
%%writefile georisk_twin_proj/src/georisk_twin/registry.py
"""Contrato de modulos del gemelo digital y registro de modulos conectados."""
from __future__ import annotations

from typing import Any, Protocol, runtime_checkable

from .schemas import ModuloInfo

# Codigos de los subproductos del ecosistema RISKTWIN usados en la demo
COD_DATOS = "URB-01"
COD_GRAFOS = "URB-02"
COD_INDICADORES = "URB-03"
COD_ANALITICO = "URB-04"


@runtime_checkable
class ModuloGemelo(Protocol):
    """Contrato que debe cumplir cualquier subproducto para conectarse al gemelo."""

    codigo: str
    nombre: str
    version: str
    estado: str  # "mock" | "real"

    def ejecutar(self, entrada: dict[str, Any]) -> dict[str, Any]:
        ...


class RegistroModulos:
    """Coleccion de modulos conectados, indexados por codigo."""

    def __init__(self) -> None:
        self._modulos: dict[str, ModuloGemelo] = {}

    def registrar(self, modulo: Any, reemplazar: bool = False) -> None:
        if not isinstance(modulo, ModuloGemelo):
            raise TypeError(
                "El objeto no cumple el contrato ModuloGemelo "
                "(requiere codigo, nombre, version, estado y ejecutar())."
            )
        if modulo.estado not in ("mock", "real"):
            raise ValueError("estado debe ser 'mock' o 'real'.")
        if modulo.codigo in self._modulos and not reemplazar:
            raise ValueError(f"El modulo {modulo.codigo} ya esta registrado (use reemplazar=True).")
        self._modulos[modulo.codigo] = modulo

    def obtener(self, codigo: str) -> ModuloGemelo:
        if codigo not in self._modulos:
            raise KeyError(f"Modulo no registrado: {codigo}")
        return self._modulos[codigo]

    def listar(self) -> list[ModuloInfo]:
        return [
            ModuloInfo(codigo=m.codigo, nombre=m.nombre, version=m.version, estado=m.estado)
            for m in self._modulos.values()
        ]

    def __contains__(self, codigo: str) -> bool:
        return codigo in self._modulos

    def __len__(self) -> int:
        return len(self._modulos)

## Paso 5 · Servicios núcleo (OE-3)
Lógica reutilizable, **independiente de FastAPI**: se puede usar desde un notebook, un script o la API. Incluye:

- **Indicadores** por sector y tipo de evento: `n_eventos × severidad media`, normalizado a [0, 1] y clasificado en Bajo/Medio/Alto.
- **Métricas de grafo** (mismas definiciones que el bloque URB-02): grado, intermediación, cercanía; combinadas con la exposición en un índice de criticidad.
- **Módulo analítico**: regresión logística multiclase con validación cruzada estratificada.
- **Pipeline integrado**: orquesta los módulos a través del registro y **mide el tiempo de cada paso**.

El pipeline vuelve a validar con Pydantic los datos que cruzan cada frontera entre módulos: ahí es donde se detectarían los desajustes entre equipos.

In [ ]:
%%writefile georisk_twin_proj/src/georisk_twin/servicios.py
"""Logica nucleo de georisk_twin (independiente de FastAPI)."""
from __future__ import annotations

from collections import Counter
from time import perf_counter
from typing import TYPE_CHECKING

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from .registry import COD_ANALITICO, COD_DATOS, COD_GRAFOS, COD_INDICADORES
from .schemas import (
    Evento, GrafoUrbano, IndicadorSector, MetricasNodo, NivelRiesgo, PrediccionSector,
    RegistroSector, RespuestaDemo, ResultadoAnalitico, ResultadoGrafo, ResumenEventos,
    SolicitudAnalitica, TipoEvento,
)

if TYPE_CHECKING:  # solo para anotaciones; evita importaciones circulares
    from .registry import RegistroModulos

SEMILLA_MODELO = 42
UMBRAL_ALTO = 0.66
UMBRAL_MEDIO = 0.33
PESO_ESTRUCTURAL = 0.5   # criticidad = 0.5 * estructural + 0.5 * exposicion
PESO_EXPOSICION = 0.5
MIN_SECTORES = 6

NOTA_DEMO = (
    "Datos sinteticos (mock). Las etiquetas Alto/Medio/Bajo se derivan de los eventos y de la "
    "criticidad del grafo: la demo prueba la integracion entre modulos, no una prediccion validada."
)


def _minmax(valores) -> np.ndarray:
    arr = np.asarray(list(valores), dtype=float)
    rango = arr.max() - arr.min()
    return np.zeros_like(arr) if rango == 0 else (arr - arr.min()) / rango


def _nivel(valor: float) -> NivelRiesgo:
    if valor >= UMBRAL_ALTO:
        return NivelRiesgo.ALTO
    if valor >= UMBRAL_MEDIO:
        return NivelRiesgo.MEDIO
    return NivelRiesgo.BAJO


# --------------------------------------------------------------- datos
def resumir_eventos(eventos: list[Evento]) -> ResumenEventos:
    fechas = [e.fecha for e in eventos]
    return ResumenEventos(
        n_eventos=len(eventos),
        fecha_min=min(fechas),
        fecha_max=max(fechas),
        por_tipo=dict(Counter(e.tipo.value for e in eventos)),
        n_sectores=len({e.sector for e in eventos if e.sector}),
    )


# ---------------------------------------------------------- indicadores
def calcular_indicadores(eventos: list[Evento]) -> list[IndicadorSector]:
    df = pd.DataFrame(
        [{"sector": e.sector or "sin_sector", "tipo": e.tipo.value, "severidad": e.severidad} for e in eventos]
    )
    g = df.groupby(["sector", "tipo"], as_index=False).agg(
        n_eventos=("severidad", "size"), severidad_media=("severidad", "mean")
    )
    g["puntaje"] = g["n_eventos"] * g["severidad_media"]
    maximo = g["puntaje"].max()
    g["puntaje_norm"] = g["puntaje"] / maximo if maximo > 0 else 0.0
    salida = [
        IndicadorSector(
            sector=r.sector,
            tipo=TipoEvento(r.tipo),
            n_eventos=int(r.n_eventos),
            severidad_media=round(float(r.severidad_media), 3),
            puntaje=round(float(r.puntaje_norm), 4),
            nivel=_nivel(float(r.puntaje_norm)),
        )
        for r in g.itertuples(index=False)
    ]
    salida.sort(key=lambda x: (-x.puntaje, x.sector, x.tipo.value))
    return salida


# --------------------------------------------------------------- grafos
def analizar_grafo(grafo: GrafoUrbano) -> ResultadoGrafo:
    G = nx.Graph()
    G.add_nodes_from(n.id for n in grafo.nodos)
    for a in grafo.arcos:
        if G.has_edge(a.origen, a.destino):  # arcos paralelos: se conserva el mas corto
            longitud = min(G[a.origen][a.destino]["length"], a.longitud_m)
        else:
            longitud = a.longitud_m
        G.add_edge(a.origen, a.destino, length=longitud)

    ids = [n.id for n in grafo.nodos]
    bet = nx.betweenness_centrality(G, weight="length", normalized=True)
    clo = nx.closeness_centrality(G, distance="length")
    grado = dict(G.degree())

    bet_n = _minmax(bet[i] for i in ids)
    clo_n = _minmax(clo[i] for i in ids)
    deg_n = _minmax(grado[i] for i in ids)
    expo_n = _minmax(n.exposicion for n in grafo.nodos)

    estructural = 0.5 * bet_n + 0.25 * clo_n + 0.25 * deg_n
    criticidad = PESO_ESTRUCTURAL * estructural + PESO_EXPOSICION * expo_n

    metricas = [
        MetricasNodo(
            id=n.id,
            sector=n.sector,
            grado=int(grado[n.id]),
            betweenness=round(float(bet[n.id]), 6),
            closeness=round(float(clo[n.id]), 6),
            exposicion_normalizada=round(float(expo_n[k]), 6),
            criticidad=round(float(criticidad[k]), 6),
            nivel=_nivel(float(criticidad[k])),
        )
        for k, n in enumerate(grafo.nodos)
    ]
    metricas.sort(key=lambda m: (-m.criticidad, m.id))
    n_comp = nx.number_connected_components(G)
    return ResultadoGrafo(
        n_nodos=G.number_of_nodes(), n_arcos=G.number_of_edges(),
        conexo=(n_comp == 1), n_componentes=n_comp, metricas=metricas,
    )


# ------------------------------------------------------------- analitico
def construir_registros_sector(
    eventos: list[Evento], grafo: GrafoUrbano, resultado: ResultadoGrafo
) -> list[RegistroSector]:
    """Une eventos (por sector) con criticidad del grafo (por sector) y etiqueta por terciles."""
    ev = pd.DataFrame([{"sector": e.sector, "severidad": e.severidad} for e in eventos if e.sector])
    if ev.empty:
        raise ValueError("Los eventos no tienen sector asignado.")
    ev = ev.groupby("sector").agg(n_eventos=("severidad", "size"), severidad_media=("severidad", "mean")).reset_index()

    crit = {m.id: m.criticidad for m in resultado.metricas}
    nodos = pd.DataFrame(
        [{"sector": n.sector, "criticidad": crit[n.id], "exposicion": n.exposicion} for n in grafo.nodos if n.sector]
    )
    if nodos.empty:
        raise ValueError("Los nodos del grafo no tienen sector asignado.")
    nodos = nodos.groupby("sector").agg(criticidad_media=("criticidad", "mean"), exposicion_media=("exposicion", "mean")).reset_index()

    df = ev.merge(nodos, on="sector", how="inner")
    if len(df) < MIN_SECTORES:
        raise ValueError(f"Se requieren al menos {MIN_SECTORES} sectores comunes entre eventos y grafo (hay {len(df)}).")

    score = 0.5 * _minmax(df["n_eventos"] * df["severidad_media"]) + 0.5 * _minmax(df["criticidad_media"])
    orden = pd.Series(score).rank(method="first")
    df["nivel"] = pd.qcut(orden, 3, labels=["Bajo", "Medio", "Alto"]).astype(str)

    return [
        RegistroSector(
            sector=r.sector, n_eventos=int(r.n_eventos),
            severidad_media=round(float(r.severidad_media), 4),
            criticidad_media=round(float(r.criticidad_media), 6),
            exposicion_media=round(float(r.exposicion_media), 4),
            nivel_riesgo=NivelRiesgo(r.nivel),
        )
        for r in df.itertuples(index=False)
    ]


def analizar_riesgo(registros: list[RegistroSector]) -> ResultadoAnalitico:
    if len(registros) < MIN_SECTORES:
        raise ValueError(f"Se requieren al menos {MIN_SECTORES} registros.")
    if any(r.nivel_riesgo is None for r in registros):
        raise ValueError("Todos los registros necesitan nivel_riesgo para entrenar.")

    X = np.array([[r.n_eventos, r.severidad_media, r.criticidad_media, r.exposicion_media] for r in registros])
    y = np.array([r.nivel_riesgo.value for r in registros])
    cuentas = Counter(y)
    if len(cuentas) < 2:
        raise ValueError("Se requieren al menos dos niveles de riesgo distintos.")

    modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEMILLA_MODELO))
    exactitud = None
    minimo = min(cuentas.values())
    if minimo >= 2:
        cv = StratifiedKFold(n_splits=min(3, minimo), shuffle=True, random_state=SEMILLA_MODELO)
        exactitud = round(float(cross_val_score(modelo, X, y, cv=cv).mean()), 4)

    modelo.fit(X, y)
    proba = modelo.predict_proba(X)
    clases = modelo.classes_
    preds = [
        PrediccionSector(
            sector=r.sector,
            nivel=NivelRiesgo(clases[int(np.argmax(p))]),
            probabilidad=round(float(np.max(p)), 4),
        )
        for r, p in zip(registros, proba)
    ]
    return ResultadoAnalitico(
        modelo="LogisticRegression + StandardScaler", n_registros=len(registros),
        exactitud_cv=exactitud, predicciones=preds,
    )


# -------------------------------------------------------------- pipeline
def ejecutar_pipeline(registro: "RegistroModulos", n_eventos: int = 300, semilla: int = 42) -> RespuestaDemo:
    """Integracion horizontal: datos -> indicadores -> grafo -> analitico, via el registro de modulos."""
    tiempos: dict[str, float] = {}

    def medir(nombre: str, fn):
        t0 = perf_counter()
        resultado = fn()
        tiempos[nombre] = round((perf_counter() - t0) * 1000, 2)
        return resultado

    mod_datos = registro.obtener(COD_DATOS)
    mod_ind = registro.obtener(COD_INDICADORES)
    mod_graf = registro.obtener(COD_GRAFOS)
    mod_ana = registro.obtener(COD_ANALITICO)

    datos = medir(f"{COD_DATOS}_datos", lambda: mod_datos.ejecutar({"n_eventos": n_eventos, "semilla": semilla}))
    eventos = [Evento.model_validate(d) for d in datos["eventos"]]          # frontera validada

    ind = medir(f"{COD_INDICADORES}_indicadores", lambda: mod_ind.ejecutar({"eventos": datos["eventos"]}))
    indicadores = [IndicadorSector.model_validate(d) for d in ind["indicadores"]]

    graf = medir(f"{COD_GRAFOS}_grafo", lambda: mod_graf.ejecutar({"semilla": semilla}))
    grafo = GrafoUrbano.model_validate(graf["grafo"])
    resultado_grafo = ResultadoGrafo.model_validate(graf["resultado"])

    registros = construir_registros_sector(eventos, grafo, resultado_grafo)
    ana = medir(
        f"{COD_ANALITICO}_analitico",
        lambda: mod_ana.ejecutar({"registros": [r.model_dump(mode="json") for r in registros]}),
    )
    analitico = ResultadoAnalitico.model_validate(ana["resultado"])

    return RespuestaDemo(
        modulos_usados=[COD_DATOS, COD_INDICADORES, COD_GRAFOS, COD_ANALITICO],
        n_eventos=len(eventos),
        n_nodos=resultado_grafo.n_nodos,
        n_sectores=len(registros),
        indicadores_top=indicadores[:5],
        nodos_criticos_top=resultado_grafo.metricas[:5],
        analitico=analitico,
        tiempos_ms=tiempos,
        nota=NOTA_DEMO,
    )

## Paso 6 · Mocks de los subproductos (Fase 4 del Kanban)
Mientras las otras tesis no entreguen su módulo, una tarjeta dependiente pasaría a *Bloqueado*. Para no detener el desarrollo se crean **mocks que cumplen el contrato**: generan datos sintéticos con la misma estructura que se espera de los módulos reales.

- Zona ficticia de 12 × 12 nodos (malla vial) dividida en 9 sectores (`S01`–`S09`), con coordenadas cercanas a Guayaquil.
- Eventos de 2023 a 2025 (tres años, el mínimo de historia que pide el borrador de datos del proyecto), con intensidad desigual por sector.
- Todo es **determinista** con una semilla: la misma semilla produce los mismos datos.

In [ ]:
%%writefile georisk_twin_proj/src/georisk_twin/mocks.py
"""Datos y modulos simulados (mocks) que respetan los contratos de georisk_twin."""
from __future__ import annotations

from datetime import datetime, timedelta
from typing import Any

import numpy as np

from .registry import COD_ANALITICO, COD_DATOS, COD_GRAFOS, COD_INDICADORES, RegistroModulos
from .schemas import (
    ArcoGrafo, Coordenada, Evento, GrafoUrbano, NodoGrafo, SolicitudAnalitica, TipoEvento,
)
from .servicios import analizar_grafo, analizar_riesgo, calcular_indicadores

LAT0, LON0, PASO = -2.2400, -79.9000, 0.0009   # origen de la zona ficticia y paso de malla (~100 m)
N_MALLA = 12
BLOQUE = 4
SECTORES_POR_LADO = N_MALLA // BLOQUE          # 3 x 3 = 9 sectores


def _sector(i: int, j: int) -> str:
    return f"S{(i // BLOQUE) * SECTORES_POR_LADO + (j // BLOQUE) + 1:02d}"


def generar_eventos_mock(n: int = 300, semilla: int = 42) -> list[Evento]:
    rng = np.random.default_rng(semilla)
    n_sec = SECTORES_POR_LADO ** 2
    pesos = rng.gamma(2.0, 1.0, n_sec)
    pesos = pesos / pesos.sum()
    inicio = datetime(2023, 1, 1)
    tipos = list(TipoEvento)
    eventos = []
    for k in range(n):
        s = int(rng.choice(n_sec, p=pesos))
        i0 = (s // SECTORES_POR_LADO) * BLOQUE
        j0 = (s % SECTORES_POR_LADO) * BLOQUE
        lat = LAT0 + (i0 + rng.uniform(0, BLOQUE - 0.01)) * PASO
        lon = LON0 + (j0 + rng.uniform(0, BLOQUE - 0.01)) * PASO
        eventos.append(
            Evento(
                id=f"EV-{k:05d}",
                fecha=inicio + timedelta(days=int(rng.integers(0, 3 * 365)), hours=int(rng.integers(0, 24))),
                tipo=tipos[int(rng.integers(0, len(tipos)))],
                ubicacion=Coordenada(lat=round(lat, 6), lon=round(lon, 6)),
                severidad=int(rng.choice([1, 2, 3, 4, 5], p=[0.35, 0.30, 0.20, 0.10, 0.05])),
                sector=f"S{s + 1:02d}",
            )
        )
    return eventos


def generar_grafo_mock(semilla: int = 42) -> GrafoUrbano:
    rng = np.random.default_rng(semilla + 1)
    expo_sector = rng.gamma(2.0, 1.0, SECTORES_POR_LADO ** 2)
    nodos, arcos = [], []
    for i in range(N_MALLA):
        for j in range(N_MALLA):
            s = (i // BLOQUE) * SECTORES_POR_LADO + (j // BLOQUE)
            nodos.append(
                NodoGrafo(
                    id=i * N_MALLA + j,
                    ubicacion=Coordenada(lat=round(LAT0 + i * PASO, 6), lon=round(LON0 + j * PASO, 6)),
                    exposicion=round(float(expo_sector[s] * rng.uniform(0.5, 1.5)), 4),
                    sector=_sector(i, j),
                )
            )
            if j + 1 < N_MALLA:
                arcos.append(ArcoGrafo(origen=i * N_MALLA + j, destino=i * N_MALLA + j + 1,
                                       longitud_m=round(float(100 * rng.uniform(0.8, 1.3)), 2)))
            if i + 1 < N_MALLA:
                arcos.append(ArcoGrafo(origen=i * N_MALLA + j, destino=(i + 1) * N_MALLA + j,
                                       longitud_m=round(float(100 * rng.uniform(0.8, 1.3)), 2)))
    return GrafoUrbano(nodos=nodos, arcos=arcos)


# ---------------------------------------------------------------- modulos mock
class ModuloDatosMock:
    codigo, nombre, version, estado = COD_DATOS, "Datos urbanos (mock)", "0.5.0-mock", "mock"

    def ejecutar(self, entrada: dict[str, Any]) -> dict[str, Any]:
        n = int(entrada.get("n_eventos", 300))
        if n < 1:
            raise ValueError("n_eventos debe ser >= 1.")
        eventos = generar_eventos_mock(n, int(entrada.get("semilla", 42)))
        return {"eventos": [e.model_dump(mode="json") for e in eventos]}


class ModuloIndicadoresMock:
    codigo, nombre, version, estado = COD_INDICADORES, "Indicadores por sector (mock)", "0.5.0-mock", "mock"

    def ejecutar(self, entrada: dict[str, Any]) -> dict[str, Any]:
        eventos = [Evento.model_validate(d) for d in entrada["eventos"]]
        return {"indicadores": [i.model_dump(mode="json") for i in calcular_indicadores(eventos)]}


class ModuloGrafosMock:
    codigo, nombre, version, estado = COD_GRAFOS, "Grafo vial y criticidad (mock)", "0.5.0-mock", "mock"

    def ejecutar(self, entrada: dict[str, Any]) -> dict[str, Any]:
        if entrada.get("grafo"):
            grafo = GrafoUrbano.model_validate(entrada["grafo"])
        else:
            grafo = generar_grafo_mock(int(entrada.get("semilla", 42)))
        return {"grafo": grafo.model_dump(mode="json"), "resultado": analizar_grafo(grafo).model_dump(mode="json")}


class ModuloAnaliticoMock:
    codigo, nombre, version, estado = COD_ANALITICO, "Clasificador de riesgo (mock)", "0.5.0-mock", "mock"

    def ejecutar(self, entrada: dict[str, Any]) -> dict[str, Any]:
        solicitud = SolicitudAnalitica.model_validate({"registros": entrada["registros"]})
        return {"resultado": analizar_riesgo(solicitud.registros).model_dump(mode="json")}


def crear_registro_mock() -> RegistroModulos:
    registro = RegistroModulos()
    for modulo in (ModuloDatosMock(), ModuloIndicadoresMock(), ModuloGrafosMock(), ModuloAnaliticoMock()):
        registro.registrar(modulo)
    return registro

## Paso 7 · API con FastAPI y especificación OpenAPI (HU-03)
Cada ruta declara su **modelo de entrada y de salida** con los contratos del Paso 3. FastAPI genera automáticamente la especificación **OpenAPI** y la documentación interactiva (`/docs`), de modo que la documentación nace del código y no puede quedar desactualizada.

Rutas principales: `/salud`, `/modulos`, `/modulos/{codigo}/ejecutar` (genérica, para que cualquier módulo nuevo sea alcanzable sin tocar la API), `/mocks/eventos`, `/datos/resumen`, `/indicadores/sector`, `/grafos/metricas`, `/analitico/riesgo` y `/demo/integracion`.

Los errores de validación devuelven **HTTP 422** con el detalle, y los módulos inexistentes **HTTP 404**.

In [ ]:
%%writefile georisk_twin_proj/src/georisk_twin/api.py
"""API REST de georisk_twin (FastAPI). La especificacion OpenAPI se genera a partir de estas rutas."""
from typing import Any, Optional

from fastapi import Body, FastAPI, HTTPException, Query

from . import __version__
from .mocks import crear_registro_mock, generar_eventos_mock
from .registry import RegistroModulos
from .schemas import (
    Evento, GrafoUrbano, IndicadorSector, ModuloInfo, RespuestaDemo, ResultadoAnalitico,
    ResultadoGrafo, ResumenEventos, SolicitudAnalitica, SolicitudDemo, SolicitudEventos,
)
from .servicios import (
    analizar_grafo, analizar_riesgo, calcular_indicadores, ejecutar_pipeline, resumir_eventos,
)

DESCRIPCION = (
    "Capa comun de integracion del gemelo digital multimodal urbano-industrial. "
    "Expone los modulos de datos, indicadores, grafos y analitica bajo contratos de datos "
    "validados con Pydantic. Los modulos reales se conectan mediante el registro de modulos."
)

ETIQUETAS = [
    {"name": "sistema", "description": "Estado del servicio y modulos conectados."},
    {"name": "datos", "description": "Eventos georreferenciados (modulo de datos)."},
    {"name": "indicadores", "description": "Indicadores de riesgo por sector."},
    {"name": "grafos", "description": "Metricas de conectividad, centralidad y criticidad."},
    {"name": "analitico", "description": "Clasificacion de nivel de riesgo."},
    {"name": "demo", "description": "Integracion horizontal de extremo a extremo."},
]


def crear_app(registro: Optional[RegistroModulos] = None) -> FastAPI:
    registro = registro if registro is not None else crear_registro_mock()
    app = FastAPI(title="georisk_twin API", version=__version__, description=DESCRIPCION, openapi_tags=ETIQUETAS)
    app.state.registro = registro

    @app.get("/salud", tags=["sistema"], summary="Estado del servicio",
             description="Devuelve 'ok' y la version del paquete si el servicio esta activo.")
    def salud() -> dict[str, str]:
        return {"estado": "ok", "version": __version__}

    @app.get("/modulos", tags=["sistema"], response_model=list[ModuloInfo], summary="Modulos conectados",
             description="Lista los modulos registrados, su version y si son mock o reales.")
    def listar_modulos() -> list[ModuloInfo]:
        return registro.listar()

    @app.post("/modulos/{codigo}/ejecutar", tags=["sistema"], summary="Ejecutar un modulo registrado",
              description="Ejecuta el modulo indicado con una entrada JSON libre y devuelve su salida. "
                          "Permite probar un modulo nuevo sin agregar rutas.")
    def ejecutar_modulo(codigo: str, entrada: dict[str, Any] = Body(...)) -> dict[str, Any]:
        try:
            modulo = registro.obtener(codigo)
        except KeyError as e:
            raise HTTPException(status_code=404, detail=e.args[0])
        try:
            return modulo.ejecutar(entrada)
        except (ValueError, KeyError) as e:
            raise HTTPException(status_code=422, detail=f"Entrada invalida para {codigo}: {e}")

    @app.get("/mocks/eventos", tags=["datos"], response_model=list[Evento], summary="Eventos sinteticos",
             description="Genera eventos de ejemplo que respetan el contrato, de forma determinista segun la semilla.")
    def mocks_eventos(n: int = Query(50, ge=1, le=2000), semilla: int = 42) -> list[Evento]:
        return generar_eventos_mock(n, semilla)

    @app.post("/datos/resumen", tags=["datos"], response_model=ResumenEventos, summary="Validar y resumir eventos",
              description="Valida una lista de eventos contra el contrato y devuelve un resumen.")
    def datos_resumen(solicitud: SolicitudEventos) -> ResumenEventos:
        return resumir_eventos(solicitud.eventos)

    @app.post("/indicadores/sector", tags=["indicadores"], response_model=list[IndicadorSector],
              summary="Indicadores por sector y tipo",
              description="Calcula n_eventos x severidad media por sector y tipo, normalizado y clasificado.")
    def indicadores_sector(solicitud: SolicitudEventos) -> list[IndicadorSector]:
        return calcular_indicadores(solicitud.eventos)

    @app.post("/grafos/metricas", tags=["grafos"], response_model=ResultadoGrafo, summary="Metricas del grafo",
              description="Calcula grado, intermediacion, cercania y un indice de criticidad que combina "
                          "estructura y exposicion.")
    def grafos_metricas(grafo: GrafoUrbano) -> ResultadoGrafo:
        return analizar_grafo(grafo)

    @app.post("/analitico/riesgo", tags=["analitico"], response_model=ResultadoAnalitico,
              summary="Clasificar riesgo por sector",
              description="Entrena una regresion logistica con los registros etiquetados y devuelve el nivel "
                          "Alto/Medio/Bajo por sector con su exactitud en validacion cruzada.")
    def analitico_riesgo(solicitud: SolicitudAnalitica) -> ResultadoAnalitico:
        try:
            return analizar_riesgo(solicitud.registros)
        except ValueError as e:
            raise HTTPException(status_code=422, detail=str(e))

    @app.post("/demo/integracion", tags=["demo"], response_model=RespuestaDemo, summary="Demo de integracion",
              description="Ejecuta datos -> indicadores -> grafo -> analitico a traves del registro de modulos "
                          "y devuelve resultados y tiempos por paso. Es determinista segun la semilla.")
    def demo_integracion(solicitud: SolicitudDemo = SolicitudDemo()) -> RespuestaDemo:
        try:
            return ejecutar_pipeline(registro, solicitud.n_eventos, solicitud.semilla)
        except KeyError as e:
            raise HTTPException(status_code=404, detail=e.args[0])
        except ValueError as e:
            raise HTTPException(status_code=422, detail=str(e))

    return app


app = crear_app()

### Instalación del paquete en el entorno
Instalación en modo editable con las dependencias de desarrollo. Además se agrega `src/` al `sys.path` para poder importar el paquete en esta misma sesión sin reiniciar el kernel.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "./georisk_twin_proj[dev]"],
                   capture_output=True, text=True)
print("pip install -e →", "OK" if r.returncode == 0 else "ERROR")
if r.returncode != 0:
    print(r.stderr[-2000:])
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath("georisk_twin_proj/src"))

import georisk_twin
print("georisk_twin", georisk_twin.__version__)

#### Exploración en memoria
`TestClient` permite llamar a la API **sin levantar un servidor**, ideal para probar rápido. (Más abajo, en la evaluación, sí se levanta un servidor real con `uvicorn`.)

In [ ]:
from fastapi.testclient import TestClient
from georisk_twin.api import crear_app

cliente = TestClient(crear_app())

print("GET /salud   →", cliente.get("/salud").json())
print("GET /modulos →")
display(pd.DataFrame(cliente.get("/modulos").json()))

eventos_json = cliente.get("/mocks/eventos", params={"n": 200, "semilla": 7}).json()
print("\nPrimer evento mock:", eventos_json[0])

resumen = cliente.post("/datos/resumen", json={"eventos": eventos_json}).json()
print("\nPOST /datos/resumen →", resumen)

# Un integrante envía un campo con otro nombre → la API lo rechaza con 422 y explica por qué
malo = [{**eventos_json[0], "gravedad": 3}]
r = cliente.post("/datos/resumen", json={"eventos": malo})
print("\nDesajuste de contrato → HTTP", r.status_code, "|", r.json()["detail"][0]["msg"], "|", r.json()["detail"][0]["loc"])

### Especificación OpenAPI
Se exporta `openapi.json` (el artefacto que consumirían los otros equipos para generar clientes) y se listan las operaciones documentadas.

In [ ]:
spec = cliente.get("/openapi.json").json()
with open("georisk_twin_proj/openapi.json", "w", encoding="utf-8") as f:
    json.dump(spec, f, ensure_ascii=False, indent=2)

filas = []
for ruta, ops in spec["paths"].items():
    for metodo, op in ops.items():
        filas.append({"método": metodo.upper(), "ruta": ruta, "resumen": op.get("summary"),
                      "descripción": bool(op.get("description")), "etiqueta": ", ".join(op.get("tags", []))})
tabla_api = pd.DataFrame(filas)
print(f"OpenAPI {spec['openapi']} · {spec['info']['title']} v{spec['info']['version']} · {len(tabla_api)} operaciones")
display(tabla_api)

doc_ok = bool(len(tabla_api) > 0 and tabla_api["resumen"].notna().all() and tabla_api["descripción"].all())
EVIDENCIA["C2_documentacion_completa"] = doc_ok
EVIDENCIA["n_endpoints"] = len(tabla_api)
print("Todas las operaciones tienen resumen y descripción:", doc_ok)

## Paso 8 · Pruebas unitarias y cobertura (HU-04)
Tres archivos de prueba, uno por capa:

| Archivo | Qué verifica |
|---|---|
| `test_schemas.py` | Los contratos **rechazan** datos inválidos (rangos, campos extra, referencias rotas). |
| `test_servicios.py` | Indicadores, métricas de grafo, analítico, registro de módulos y reproducibilidad del pipeline. |
| `test_api.py` | Códigos HTTP (200/404/422), contenido de las respuestas y que OpenAPI esté completo. |

In [ ]:
%%writefile georisk_twin_proj/tests/test_schemas.py
import pytest
from pydantic import ValidationError

from georisk_twin.schemas import (
    ArcoGrafo, Coordenada, Evento, GrafoUrbano, NodoGrafo, RegistroSector, SolicitudAnalitica,
)

EVENTO = {"id": "EV-1", "fecha": "2025-01-01T10:00:00", "tipo": "accidente",
          "ubicacion": {"lat": -2.2, "lon": -79.9}, "severidad": 2, "sector": "S01"}


def _nodo(i):
    return NodoGrafo(id=i, ubicacion=Coordenada(lat=-2.2, lon=-79.9))


def test_evento_valido():
    assert Evento(**EVENTO).severidad == 2


def test_coordenada_fuera_de_rango():
    with pytest.raises(ValidationError):
        Coordenada(lat=95, lon=0)


def test_evento_rechaza_campos_extra():
    with pytest.raises(ValidationError):
        Evento(**{**EVENTO, "gravedad": 3})


def test_evento_rechaza_tipo_desconocido():
    with pytest.raises(ValidationError):
        Evento(**{**EVENTO, "tipo": "incendio"})


def test_severidad_fuera_de_rango():
    with pytest.raises(ValidationError):
        Evento(**{**EVENTO, "severidad": 9})


def test_arco_autorreferencia():
    with pytest.raises(ValidationError):
        ArcoGrafo(origen=1, destino=1, longitud_m=10)


def test_arco_a_nodo_inexistente():
    with pytest.raises(ValidationError):
        GrafoUrbano(nodos=[_nodo(0), _nodo(1)], arcos=[ArcoGrafo(origen=0, destino=5, longitud_m=10)])


def test_nodos_duplicados():
    with pytest.raises(ValidationError):
        GrafoUrbano(nodos=[_nodo(0), _nodo(0)], arcos=[ArcoGrafo(origen=0, destino=1, longitud_m=10)])


def test_solicitud_analitica_requiere_minimo_de_registros():
    reg = RegistroSector(sector="S", n_eventos=1, severidad_media=1, criticidad_media=0.5, exposicion_media=1)
    with pytest.raises(ValidationError):
        SolicitudAnalitica(registros=[reg] * 3)

In [ ]:
%%writefile georisk_twin_proj/tests/test_servicios.py
import pytest

from georisk_twin.mocks import (
    ModuloDatosMock, crear_registro_mock, generar_eventos_mock, generar_grafo_mock,
)
from georisk_twin.registry import RegistroModulos
from georisk_twin.schemas import (
    ArcoGrafo, Coordenada, GrafoUrbano, NodoGrafo, NivelRiesgo, RegistroSector,
)
from georisk_twin.servicios import (
    analizar_grafo, analizar_riesgo, calcular_indicadores, construir_registros_sector,
    ejecutar_pipeline, resumir_eventos,
)


def _camino(n=5):
    nodos = [NodoGrafo(id=i, ubicacion=Coordenada(lat=-2.2, lon=-79.9)) for i in range(n)]
    arcos = [ArcoGrafo(origen=i, destino=i + 1, longitud_m=100) for i in range(n - 1)]
    return GrafoUrbano(nodos=nodos, arcos=arcos)


def test_mocks_son_deterministas():
    a = [e.model_dump() for e in generar_eventos_mock(50, 1)]
    b = [e.model_dump() for e in generar_eventos_mock(50, 1)]
    c = [e.model_dump() for e in generar_eventos_mock(50, 2)]
    assert a == b and a != c


def test_resumen_de_eventos():
    r = resumir_eventos(generar_eventos_mock(120, 3))
    assert r.n_eventos == 120 and sum(r.por_tipo.values()) == 120 and r.n_sectores <= 9


def test_indicadores_normalizados():
    ind = calcular_indicadores(generar_eventos_mock(300, 4))
    assert ind[0].puntaje == 1.0
    assert all(0 <= i.puntaje <= 1 for i in ind)
    assert sum(i.n_eventos for i in ind) == 300


def test_grafo_camino_el_centro_es_el_mas_critico():
    res = analizar_grafo(_camino(5))
    assert res.conexo and res.n_componentes == 1
    assert res.metricas[0].id == 2
    assert all(0 <= m.criticidad <= 1 for m in res.metricas)


def test_grafo_desconectado():
    nodos = [NodoGrafo(id=i, ubicacion=Coordenada(lat=-2.2, lon=-79.9)) for i in range(4)]
    arcos = [ArcoGrafo(origen=0, destino=1, longitud_m=10), ArcoGrafo(origen=2, destino=3, longitud_m=10)]
    res = analizar_grafo(GrafoUrbano(nodos=nodos, arcos=arcos))
    assert not res.conexo and res.n_componentes == 2


def test_arcos_paralelos_conservan_el_mas_corto():
    nodos = [NodoGrafo(id=i, ubicacion=Coordenada(lat=-2.2, lon=-79.9)) for i in range(2)]
    arcos = [ArcoGrafo(origen=0, destino=1, longitud_m=50), ArcoGrafo(origen=1, destino=0, longitud_m=30)]
    assert analizar_grafo(GrafoUrbano(nodos=nodos, arcos=arcos)).n_arcos == 1


def test_analitico_con_datos_mock():
    ev = generar_eventos_mock(300, 5)
    g = generar_grafo_mock(5)
    regs = construir_registros_sector(ev, g, analizar_grafo(g))
    assert len(regs) == 9
    res = analizar_riesgo(regs)
    assert len(res.predicciones) == 9
    assert res.exactitud_cv is None or 0 <= res.exactitud_cv <= 1


def test_analitico_pocos_registros():
    reg = RegistroSector(sector="S", n_eventos=1, severidad_media=1, criticidad_media=0.5,
                         exposicion_media=1, nivel_riesgo=NivelRiesgo.BAJO)
    with pytest.raises(ValueError):
        analizar_riesgo([reg] * 3)


def test_analitico_sin_etiquetas():
    reg = RegistroSector(sector="S", n_eventos=1, severidad_media=1, criticidad_media=0.5, exposicion_media=1)
    with pytest.raises(ValueError):
        analizar_riesgo([reg] * 6)


def test_analitico_una_sola_clase():
    reg = RegistroSector(sector="S", n_eventos=1, severidad_media=1, criticidad_media=0.5,
                         exposicion_media=1, nivel_riesgo=NivelRiesgo.ALTO)
    with pytest.raises(ValueError):
        analizar_riesgo([reg] * 6)


def test_registro_rechaza_objeto_que_no_cumple_el_contrato():
    with pytest.raises(TypeError):
        RegistroModulos().registrar(object())


def test_registro_rechaza_duplicados_y_permite_reemplazar():
    r = RegistroModulos()
    r.registrar(ModuloDatosMock())
    with pytest.raises(ValueError):
        r.registrar(ModuloDatosMock())
    r.registrar(ModuloDatosMock(), reemplazar=True)
    assert len(r) == 1 and "URB-01" in r


def test_registro_modulo_inexistente():
    with pytest.raises(KeyError):
        RegistroModulos().obtener("URB-99")


def test_pipeline_integra_los_cuatro_modulos():
    r = ejecutar_pipeline(crear_registro_mock(), n_eventos=200, semilla=7)
    assert r.modulos_usados == ["URB-01", "URB-03", "URB-02", "URB-04"]
    assert r.n_eventos == 200 and r.n_nodos == 144 and r.n_sectores == 9
    assert len(r.tiempos_ms) == 4


def test_pipeline_es_reproducible():
    a = ejecutar_pipeline(crear_registro_mock(), 150, 11).model_dump(exclude={"tiempos_ms"})
    b = ejecutar_pipeline(crear_registro_mock(), 150, 11).model_dump(exclude={"tiempos_ms"})
    c = ejecutar_pipeline(crear_registro_mock(), 150, 12).model_dump(exclude={"tiempos_ms"})
    assert a == b and a != c

In [ ]:
%%writefile georisk_twin_proj/tests/test_api.py
import pytest

pytest.importorskip("fastapi")
pytest.importorskip("httpx")

from fastapi.testclient import TestClient  # noqa: E402

from georisk_twin.api import crear_app  # noqa: E402

cliente = TestClient(crear_app())

GRAFO = {
    "nodos": [{"id": i, "ubicacion": {"lat": -2.2, "lon": -79.9}, "exposicion": 1.0} for i in range(5)],
    "arcos": [{"origen": i, "destino": i + 1, "longitud_m": 100} for i in range(4)],
}


def test_salud():
    r = cliente.get("/salud")
    assert r.status_code == 200 and r.json()["estado"] == "ok"


def test_modulos():
    r = cliente.get("/modulos")
    assert r.status_code == 200
    assert {m["codigo"] for m in r.json()} == {"URB-01", "URB-02", "URB-03", "URB-04"}


def test_mocks_eventos_y_resumen():
    eventos = cliente.get("/mocks/eventos", params={"n": 30, "semilla": 1}).json()
    assert len(eventos) == 30
    r = cliente.post("/datos/resumen", json={"eventos": eventos})
    assert r.status_code == 200 and r.json()["n_eventos"] == 30


def test_resumen_vacio_es_422():
    assert cliente.post("/datos/resumen", json={"eventos": []}).status_code == 422


def test_campo_extra_es_422():
    eventos = cliente.get("/mocks/eventos", params={"n": 1}).json()
    eventos[0]["gravedad"] = 3
    assert cliente.post("/datos/resumen", json={"eventos": eventos}).status_code == 422


def test_indicadores():
    eventos = cliente.get("/mocks/eventos", params={"n": 100}).json()
    r = cliente.post("/indicadores/sector", json={"eventos": eventos})
    assert r.status_code == 200 and r.json()[0]["puntaje"] == 1.0


def test_grafo_metricas():
    r = cliente.post("/grafos/metricas", json=GRAFO)
    assert r.status_code == 200
    assert r.json()["metricas"][0]["id"] == 2


def test_grafo_invalido_es_422():
    malo = {**GRAFO, "arcos": [{"origen": 0, "destino": 99, "longitud_m": 10}]}
    assert cliente.post("/grafos/metricas", json=malo).status_code == 422


def test_analitico_pocos_registros_es_422():
    assert cliente.post("/analitico/riesgo", json={"registros": []}).status_code == 422


def test_demo_integracion():
    r = cliente.post("/demo/integracion", json={"n_eventos": 120, "semilla": 3})
    assert r.status_code == 200
    cuerpo = r.json()
    assert cuerpo["modulos_usados"] == ["URB-01", "URB-03", "URB-02", "URB-04"]
    assert "nota" in cuerpo


def test_demo_con_cuerpo_vacio_usa_valores_por_defecto():
    assert cliente.post("/demo/integracion").status_code == 200


def test_ejecutar_modulo_generico():
    r = cliente.post("/modulos/URB-01/ejecutar", json={"n_eventos": 5, "semilla": 1})
    assert r.status_code == 200 and len(r.json()["eventos"]) == 5


def test_modulo_inexistente_es_404():
    assert cliente.post("/modulos/URB-99/ejecutar", json={}).status_code == 404


def test_modulo_con_entrada_invalida_es_422():
    assert cliente.post("/modulos/URB-03/ejecutar", json={}).status_code == 422


def test_openapi_documenta_todas_las_operaciones():
    spec = cliente.get("/openapi.json").json()
    for ruta, ops in spec["paths"].items():
        for metodo, op in ops.items():
            assert op.get("summary") and op.get("description"), f"{metodo} {ruta} sin documentar"

In [ ]:
r = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", f"--cov=georisk_twin", "--cov-report=term-missing",
     "--cov-report=json:cov.json", f"--cov-fail-under={META_COBERTURA_PCT}"],
    cwd="georisk_twin_proj", capture_output=True, text=True)
print(r.stdout[-4500:])
if r.returncode != 0:
    print(r.stderr[-1500:])

with open("georisk_twin_proj/cov.json") as f:
    cov = json.load(f)["totals"]["percent_covered"]
EVIDENCIA["tests_pasan"] = (r.returncode == 0)
EVIDENCIA["cobertura_pct"] = round(cov, 1)
print(f"\nPruebas: {'✔ todas pasan' if r.returncode == 0 else '✘ hay fallos'} · cobertura {cov:.1f} % (meta {META_COBERTURA_PCT} %)")

## Paso 9 · Demo de integración horizontal (HU-06)
Es el **umbral mínimo** del anteproyecto: conectar datos, indicadores y un módulo analítico (aquí también el grafo). La demo se ejecuta a través del registro de módulos y de la API, y muestra tiempos por paso.

Cada salida cruza una frontera validada por Pydantic: si un módulo real entregara datos con otra forma, la demo fallaría en ese punto con un mensaje explícito.

In [ ]:
resp = cliente.post("/demo/integracion", json={"n_eventos": 300, "semilla": 42})
assert resp.status_code == 200, resp.text
demo = resp.json()

print("Módulos conectados:", " → ".join(demo["modulos_usados"]))
print(f"Eventos: {demo['n_eventos']} | Nodos del grafo: {demo['n_nodos']} | Sectores: {demo['n_sectores']}")
print("\nIndicadores: sectores/tipos con mayor puntaje")
display(pd.DataFrame(demo["indicadores_top"]))
print("Nodos más críticos del grafo")
display(pd.DataFrame(demo["nodos_criticos_top"])[["id", "sector", "grado", "betweenness", "exposicion_normalizada", "criticidad", "nivel"]])
print("Analítico:", demo["analitico"]["modelo"], "| exactitud CV:", demo["analitico"]["exactitud_cv"])
display(pd.DataFrame(demo["analitico"]["predicciones"]))
print("Tiempos por paso (ms):", demo["tiempos_ms"])
print("\n⚠", demo["nota"])

EVIDENCIA["C4_conexion_minima"] = {"URB-01", "URB-03", "URB-04"} <= set(demo["modulos_usados"])

## Paso 10 · Conectar un módulo nuevo (plantilla de contribución, HU-05)
La prueba de fuego del diseño: ¿puede **otro equipo** sumar su módulo sin modificar el paquete? Se simula con un módulo de **simulación (URB-05)** que retira los nodos más críticos y mide cuánto se degrada la red. Solo hay que cumplir el contrato (`codigo`, `nombre`, `version`, `estado`, `ejecutar`), registrarlo y ya es alcanzable por la API genérica.

In [ ]:
%%writefile georisk_twin_proj/CONTRIBUTING.md
# Como conectar un modulo a georisk_twin

1. **Disena contra el contrato.** Los datos que recibes y entregas deben respetar los modelos de
   `georisk_twin.schemas` (usa `Modelo.model_validate(...)` al recibir y `model_dump(mode="json")` al entregar).
2. **Implementa el contrato de modulo**: una clase con los atributos `codigo`, `nombre`, `version`,
   `estado` ("mock" o "real") y el metodo `ejecutar(self, entrada: dict) -> dict`.
3. **Registralo**:

       from georisk_twin.registry import RegistroModulos
       registro.registrar(MiModulo())                    # nuevo
       registro.registrar(MiModulo(), reemplazar=True)   # reemplaza a un mock con el mismo codigo

4. **Pruebalo** con `POST /modulos/{codigo}/ejecutar` y agrega pruebas en `tests/`.
5. **Antes de abrir el pull request**: `pytest --cov=georisk_twin` debe pasar (CI en GitHub Actions)
   y otro integrante debe revisar el codigo.

Reglas: no cambies contratos existentes sin acordarlo con los demas equipos; si necesitas un campo
nuevo, agregalo como opcional.

In [ ]:
import networkx as nx
from georisk_twin.schemas import GrafoUrbano
from georisk_twin.servicios import analizar_grafo
from georisk_twin.mocks import crear_registro_mock, generar_grafo_mock


class ModuloSimulacionDemo:
    """URB-05 (demo): simula la falla de los k nodos más críticos y mide el impacto en la red."""
    codigo = "URB-05"
    nombre = "Simulación de falla de nodos (demo)"
    version = "0.1.0-demo"
    estado = "mock"

    def ejecutar(self, entrada):
        grafo = GrafoUrbano.model_validate(entrada["grafo"])
        k = int(entrada.get("k", 5))
        quitar = [m.id for m in analizar_grafo(grafo).metricas[:k]]
        G = nx.Graph()
        G.add_nodes_from(n.id for n in grafo.nodos)
        G.add_edges_from((a.origen, a.destino) for a in grafo.arcos)
        base = nx.global_efficiency(G)
        H = G.copy(); H.remove_nodes_from(quitar)
        gigante = max(nx.connected_components(H), key=len)
        return {"nodos_removidos": quitar,
                "perdida_eficiencia": round(1 - nx.global_efficiency(H) / base, 4),
                "fraccion_componente_gigante": round(len(gigante) / G.number_of_nodes(), 4)}


registro_ext = crear_registro_mock()
registro_ext.registrar(ModuloSimulacionDemo())          # ← única línea de integración
cliente_ext = TestClient(crear_app(registro_ext))

print("Módulos ahora:", [m["codigo"] for m in cliente_ext.get("/modulos").json()])
grafo_json = generar_grafo_mock(42).model_dump(mode="json")
for k in (3, 10, 20):
    out = cliente_ext.post("/modulos/URB-05/ejecutar", json={"grafo": grafo_json, "k": k}).json()
    print(f"k={k:>2} → pérdida de eficiencia {out['perdida_eficiencia']:.1%} | componente gigante {out['fraccion_componente_gigante']:.1%}")

# Y si el módulo recibe un grafo mal formado, el contrato lo detiene
malo = cliente_ext.post("/modulos/URB-05/ejecutar", json={"grafo": {"nodos": [], "arcos": []}})
print("Grafo inválido → HTTP", malo.status_code)

## Paso 11 · Evaluación del artefacto (OE-4)
La fase de evaluación de DSR exige verificar los criterios con evidencia. Se miden cuatro cosas:

1. **Instalación limpia (C1):** se crea un entorno virtual **vacío**, se instala el paquete (no editable) y se importa.
2. **Rendimiento (OE-4):** se levanta la API con `uvicorn` y se miden los tiempos de respuesta por HTTP real (mediana y p95).
3. **Reproducibilidad (C3):** misma semilla ⇒ mismo resultado; semilla distinta ⇒ resultado distinto.
4. **Documentación (C2):** ya medida en el Paso 7.

### 11.1 Instalación en entorno limpio

In [ ]:
import venv, shutil
VENV = "/tmp/venv_limpio"
shutil.rmtree(VENV, ignore_errors=True)

t0 = time.time()
venv.create(VENV, with_pip=True)
inst = subprocess.run([f"{VENV}/bin/pip", "install", "-q", os.path.abspath("georisk_twin_proj")],
                      capture_output=True, text=True)
t_inst = time.time() - t0

chk = subprocess.run([f"{VENV}/bin/python", "-c",
                      "import georisk_twin, georisk_twin.api as a; print(georisk_twin.__version__, a.app.title)"],
                     capture_output=True, text=True, cwd="/tmp")
ok_inst = (inst.returncode == 0 and chk.returncode == 0)
print("Instalación en venv vacío:", "✔ OK" if ok_inst else "✘ ERROR", f"({t_inst:.0f} s)")
print("Importación desde el venv:", chk.stdout.strip() or chk.stderr[-800:])
if inst.returncode != 0:
    print(inst.stderr[-1500:])

EVIDENCIA["C1_instalable"] = ok_inst
EVIDENCIA["tiempo_instalacion_s"] = round(t_inst)

### 11.2 Tiempo de respuesta con un servidor real
Se levanta `uvicorn` en segundo plano y se llama a cada endpoint 30 veces por HTTP (con una primera llamada de calentamiento descartada). La medición incluye serialización JSON y red local, y se hace en una máquina de Colab, así que **los valores absolutos dependen del entorno**; sirven para comparar versiones del paquete entre iteraciones.

In [ ]:
import requests, numpy as np
from georisk_twin.mocks import generar_eventos_mock, generar_grafo_mock
from georisk_twin.servicios import analizar_grafo, construir_registros_sector

PUERTO = 8765
entorno = {**os.environ, "PYTHONPATH": os.path.abspath("georisk_twin_proj/src")}
servidor = subprocess.Popen([sys.executable, "-m", "uvicorn", "georisk_twin.api:app", "--port", str(PUERTO), "--log-level", "warning"],
                            env=entorno, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
BASE = f"http://127.0.0.1:{PUERTO}"
for _ in range(40):
    try:
        if requests.get(BASE + "/salud", timeout=1).status_code == 200:
            break
    except Exception:
        time.sleep(0.5)
else:
    servidor.terminate()
    raise RuntimeError("El servidor no arrancó:\n" + (servidor.stdout.read().decode()[-1500:] if servidor.stdout else ""))
print("Servidor activo en", BASE)

# Cargas útiles de prueba
ev = generar_eventos_mock(200, 1)
gr = generar_grafo_mock(1)
regs = construir_registros_sector(ev, gr, analizar_grafo(gr))
ev_j = [e.model_dump(mode="json") for e in ev]
gr_j = gr.model_dump(mode="json")
regs_j = [r.model_dump(mode="json") for r in regs]

PRUEBAS = [
    ("GET  /salud", "get", "/salud", None),
    ("GET  /modulos", "get", "/modulos", None),
    ("GET  /mocks/eventos (n=50)", "get", "/mocks/eventos?n=50", None),
    ("POST /datos/resumen (200 ev.)", "post", "/datos/resumen", {"eventos": ev_j}),
    ("POST /indicadores/sector (200 ev.)", "post", "/indicadores/sector", {"eventos": ev_j}),
    ("POST /grafos/metricas (144 nodos)", "post", "/grafos/metricas", gr_j),
    ("POST /analitico/riesgo (9 sectores)", "post", "/analitico/riesgo", {"registros": regs_j}),
    ("POST /demo/integracion (300 ev.)", "post", "/demo/integracion", {"n_eventos": 300, "semilla": 42}),
]
N_REP = 30
filas = []
for nombre, metodo, ruta, cuerpo in PRUEBAS:
    llamar = (lambda: requests.get(BASE + ruta)) if metodo == "get" else (lambda: requests.post(BASE + ruta, json=cuerpo))
    llamar()                                   # calentamiento
    ms, codigos = [], set()
    for _ in range(N_REP):
        t = time.perf_counter(); r = llamar(); ms.append((time.perf_counter() - t) * 1000); codigos.add(r.status_code)
    filas.append({"endpoint": nombre, "HTTP": ",".join(map(str, sorted(codigos))),
                  "mediana_ms": round(np.median(ms), 1), "p95_ms": round(np.percentile(ms, 95), 1), "máx_ms": round(max(ms), 1)})

servidor.terminate()
bench = pd.DataFrame(filas)
bench["cumple_meta"] = bench["p95_ms"] <= META_P95_MS
display(bench)

EVIDENCIA["latencia_p95_max_ms"] = float(bench["p95_ms"].max())
EVIDENCIA["latencia_ok"] = bool(bench["cumple_meta"].all() and (bench["HTTP"] == "200").all())
print(f"p95 máximo: {EVIDENCIA['latencia_p95_max_ms']} ms (meta {META_P95_MS} ms)")

### 11.3 Reproducibilidad de la demo
Se ejecuta la demo varias veces, **en procesos independientes**, con la misma semilla. Los resultados (excluyendo los tiempos, que varían) deben ser idénticos; con otra semilla deben cambiar.

In [ ]:
import hashlib
script = (
    "import json, hashlib;"
    "from georisk_twin.mocks import crear_registro_mock;"
    "from georisk_twin.servicios import ejecutar_pipeline;"
    "import sys; s=int(sys.argv[1]);"
    "r=ejecutar_pipeline(crear_registro_mock(),300,s).model_dump(mode='json',exclude={'tiempos_ms'});"
    "print(hashlib.sha256(json.dumps(r,sort_keys=True).encode()).hexdigest())"
)
def huella(semilla):
    out = subprocess.run([sys.executable, "-c", script, str(semilla)], env=entorno, capture_output=True, text=True)
    assert out.returncode == 0, out.stderr[-1000:]
    return out.stdout.strip()

h42 = [huella(42) for _ in range(3)]
h7 = huella(7)
print("Semilla 42 (3 procesos):", [h[:12] for h in h42])
print("Semilla  7             :", h7[:12])

repro_ok = (len(set(h42)) == 1) and (h7 != h42[0])
EVIDENCIA["C3_reproducible"] = repro_ok
print("Reproducible:", "✔" if repro_ok else "✘")

## Paso 12 · Resultado de la iteración y CI
### 12.1 Verificación de los criterios de aceptación
Se contrastan las evidencias medidas con los cuatro criterios. Si alguno falla, el anteproyecto indica que el artefacto **no cierra la iteración v0.5 y se vuelve a Diseño y desarrollo**.

In [ ]:
cov = EVIDENCIA.get("cobertura_pct")
filas = [
    ("C1", CRITERIOS_ACEPTACION["C1"], EVIDENCIA.get("C1_instalable"),
     f"instalación en venv vacío en {EVIDENCIA.get('tiempo_instalacion_s', '?')} s"),
    ("C2", CRITERIOS_ACEPTACION["C2"], EVIDENCIA.get("C2_documentacion_completa"),
     f"{EVIDENCIA.get('n_endpoints', '?')} operaciones con resumen y descripción"),
    ("C3", CRITERIOS_ACEPTACION["C3"], EVIDENCIA.get("C3_reproducible"),
     "misma semilla ⇒ misma huella en procesos independientes"),
    ("C4", CRITERIOS_ACEPTACION["C4"], EVIDENCIA.get("C4_conexion_minima"),
     "URB-01 (datos) + URB-03 (indicadores) + URB-04 (analítico) + URB-02 (grafo)"),
]
extras = [
    ("Calidad", f"Pruebas pasan y cobertura ≥ {META_COBERTURA_PCT} %",
     bool(EVIDENCIA.get("tests_pasan")) and cov is not None and cov >= META_COBERTURA_PCT, f"cobertura {cov} %"),
    ("Desempeño", f"p95 ≤ {META_P95_MS} ms en todos los endpoints", EVIDENCIA.get("latencia_ok"),
     f"p95 máximo {EVIDENCIA.get('latencia_p95_max_ms', '?')} ms"),
]
tabla = pd.DataFrame(
    [(a, b, "✔ cumple" if c else ("✘ no cumple" if c is False else "— sin evidencia"), d) for a, b, c, d in filas + extras],
    columns=["ID", "Criterio", "Estado", "Evidencia"],
)
display(tabla)

obligatorios = [c for _, _, c, _ in filas]
if all(obligatorios):
    print("✔ Iteración v0.5 (piloto) CERRADA: se cumplen los cuatro criterios de aceptación.")
else:
    print("✘ Iteración NO cerrada: volver a Diseño y desarrollo para corregir los criterios pendientes.")

### 12.2 Integración continua (GitHub Actions)
La Fase 6 del Kanban dice que una tarjeta solo pasa a *Hecho* si las pruebas pasan **automáticamente** en cada cambio. Este archivo ejecuta `pytest` con cobertura en tres versiones de Python cada vez que se hace `push` o `pull request`. Copia la carpeta `georisk_twin_proj` a tu repositorio y el flujo queda activo.

In [ ]:
%%writefile georisk_twin_proj/.github/workflows/ci.yml
name: CI

on:
  push:
  pull_request:

jobs:
  pruebas:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - name: Instalar paquete y dependencias de desarrollo
        run: pip install -e ".[dev]"
      - name: Pruebas con cobertura minima
        run: pytest --cov=georisk_twin --cov-report=term-missing --cov-fail-under=80

### 12.3 Entregables
Se empaqueta el proyecto (sin cachés ni archivos temporales) para subirlo al repositorio y descargarlo desde Colab.

In [ ]:
import shutil
for raiz, dirs, _ in os.walk("georisk_twin_proj"):
    for d in list(dirs):
        if d in ("__pycache__", ".pytest_cache", "georisk_twin.egg-info", ".mypy_cache"):
            shutil.rmtree(os.path.join(raiz, d), ignore_errors=True); dirs.remove(d)
for f in ("georisk_twin_proj/.coverage", "georisk_twin_proj/cov.json"):
    if os.path.exists(f):
        os.remove(f)

shutil.make_archive("georisk_twin_proj", "zip", ".", "georisk_twin_proj")
print("Archivos del proyecto:")
for raiz, _, archivos in sorted(os.walk("georisk_twin_proj")):
    for a in sorted(archivos):
        print("  ", os.path.join(raiz, a))
try:
    from google.colab import files
    files.download("georisk_twin_proj.zip")
except ImportError:
    print("\nFuera de Colab: el .zip quedó en el directorio actual.")

## Conclusiones del piloto y siguientes pasos

**Qué demuestra este piloto**
- Un paquete **instalable** (`pip install .`) con contratos de datos Pydantic que **rechazan desajustes entre equipos** con mensajes claros (HTTP 422).
- Una API **documentada automáticamente** (OpenAPI) con una ruta genérica que permite probar cualquier módulo nuevo sin modificar el paquete.
- Una **integración horizontal reproducible** de datos, indicadores, grafo y analítica, con tiempo medido por paso.
- Un flujo de calidad (pytest, cobertura, CI) que convierte cada tarjeta cerrada en una pieza verificada.

**Limitaciones que conviene declarar en la tesis**
- Todos los módulos son *mocks* con datos sintéticos: la integración está demostrada, el **riesgo real no**.
- Las metas de cobertura y latencia son propuestas del piloto; los tiempos dependen de la máquina de Colab.
- La API guarda estado solo en memoria (no hay base de datos ni autenticación); un despliegue real queda fuera del alcance, como indica el anteproyecto.
- Los módulos `SI-01`, `SI-02` e `IND-SIS-01` aún no tienen rol asignado en el registro.

**Siguientes pasos**
1. Acordar con cada equipo los campos de su contrato y reemplazar los mocks por módulos reales (`registrar(..., reemplazar=True)` con `estado="real"`); por ejemplo, un adaptador que lea los CSV de nodos y aristas que entrega URB-02.
2. Subir `georisk_twin_proj` a GitHub, activar el CI y crear el tablero en Trello con las tarjetas de la tabla del Paso 1.
3. Agregar pruebas de integración con los datasets reales a medida que se entreguen, y comparar los tiempos de respuesta entre iteraciones.
4. Publicar `README.md`, `CONTRIBUTING.md` y `openapi.json` en GitHub Pages como guía técnica.